# Future studies — the two open directions, made runnable

Companion to the [main experiments notebook](https://colab.research.google.com/github/alexmarinho/IG/blob/master/studio/notebooks/iterated-greedy-experiments.ipynb).
It executes, live, the two experimental rows of the project's future-work section:

1. **GPU replica fleet** (`gpu/fleet_torch.py`) — thousands of independent Iterated Greedy
   searches advancing in lockstep as one PyTorch program (CUDA when present, CPU otherwise).
2. **LLM heuristic factory** (`factory/`) — an evolutionary loop over the *removal* operator,
   scored on a training split and re-scored on held-out seeds and instances.

Everything runs on a free Colab runtime (CPU or T4) and locally. Nothing is cached: every
cell recomputes, the notebook is safe to `Restart & Run All`, and each code cell guards on
the bootstrap flag.

**What this notebook is.** Evidence of infrastructure, not a claim of superiority. The fleet
is still slower than one optimized CPU core for a single instance; the evolved operators do
not yet beat uniform random removal on a held-out split. Both results are reproduced here
rather than asserted, because a negative result that reproduces is worth more than a
positive one that does not.


## 1. Set up the environment

Restart-safe: detects Colab or a local checkout, locates (or clones) the repository, installs
`torch`/`matplotlib` only if missing, imports the engine and defines the presentation
helpers. Sets `IG_FUTURE_READY`, which every later cell checks.


In [ ]:
import os, sys, subprocess, time, json, random, statistics, platform, html
from pathlib import Path

_T0 = time.perf_counter()
TIMINGS = {}

IN_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG'))
if not IN_COLAB:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        pass
    else:
        IN_COLAB = True

REPOSITORY_URL = 'https://github.com/alexmarinho/IG.git'
_default = '/content/IG' if IN_COLAB else str(Path.cwd())
ROOT = Path(os.environ.get('IG_REPO', _default))
if not (ROOT / 'python' / 'ig_scheduler.py').is_file():          # walk up from a subdir
    for parent in list(Path.cwd().resolve().parents)[:4]:
        if (parent / 'python' / 'ig_scheduler.py').is_file():
            ROOT = parent
            break
if not (ROOT / 'python' / 'ig_scheduler.py').is_file():
    if not IN_COLAB:
        raise FileNotFoundError('Set IG_REPO to an IG checkout, or run inside one.')
    print(f'Cloning {REPOSITORY_URL} into {ROOT} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(ROOT)], check=True)

try:
    import torch
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'torch'], check=True)
    import importlib; importlib.invalidate_caches(); import torch
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'matplotlib>=3.8,<4'], check=True)
    import importlib; importlib.invalidate_caches(); import matplotlib.pyplot as plt

from IPython.display import HTML, Markdown, display

if str(ROOT / 'python') not in sys.path:
    sys.path.insert(0, str(ROOT / 'python'))
import ig_scheduler
from ig_scheduler import Instance, solve

benchmark = json.loads((ROOT / 'benchmark.json').read_text())

PALETTE = {'ink': '#17233c', 'muted': '#687386', 'blue': '#3167c6', 'cyan': '#35a7b7',
           'green': '#27866f', 'orange': '#d77924', 'red': '#bd4b54', 'light': '#d9e1ec'}
plt.rcParams.update({'figure.dpi': 125, 'savefig.dpi': 160, 'font.size': 10,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.labelsize': 10, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': PALETTE['muted'], 'axes.labelcolor': PALETTE['ink'],
    'xtick.color': PALETTE['muted'], 'ytick.color': PALETTE['muted'],
    'text.color': PALETTE['ink'], 'legend.frameon': False})


def num(value, digits=2):
    return f'{float(value):,.{digits}f}'


def display_table(rows, *, caption=None, columns=None):
    rows = list(rows)
    if not rows:
        display(Markdown('_No rows._')); return
    columns = list(columns or rows[0].keys())
    out = ['<table style="border-collapse:collapse;width:100%;font-size:0.92rem">']
    if caption:
        out.append(f'<caption style="text-align:left;font-weight:600;padding:0 0 .5rem">{html.escape(caption)}</caption>')
    out.append('<thead><tr>')
    out += [f'<th style="text-align:left;border-bottom:2px solid #9aa6b8;padding:.38rem">{html.escape(str(c))}</th>' for c in columns]
    out.append('</tr></thead><tbody>')
    for row in rows:
        out.append('<tr>')
        for c in columns:
            v = row.get(c, '')
            align = 'right' if isinstance(v, (int, float)) else 'left'
            out.append(f'<td style="text-align:{align};border-bottom:1px solid #d9e1ec;padding:.38rem">{html.escape(str(v))}</td>')
        out.append('</tr>')
    out.append('</tbody></table>')
    display(HTML(''.join(out)))


META_SEED = 20_150_718
IG_FUTURE_READY = True
TIMINGS['setup'] = time.perf_counter() - _T0

print(f"Environment: {'Google Colab' if IN_COLAB else 'local'}")
print(f'Repository:  {ROOT}')
print(f'Engine:      ig_scheduler {ig_scheduler.__version__} · Python {platform.python_version()}')
print(f'PyTorch:     {torch.__version__} · CUDA available: {torch.cuda.is_available()}')


## 2. Direction A — GPU replica fleet

Instead of making *one* search smarter, advance **thousands of independent searches in
lockstep** over the same instance. `gpu/fleet_torch.py` keeps R replicas in one tensor
program: batched binomial removal with a per-replica ramp, a batched greedy repair that
prices *every* insertion position for *all* replicas in a single scan, and per-replica
incumbents. It runs on any CUDA GPU and, slowly, on CPU with identical numbers.

**What to expect.** The fleet trades single-trajectory depth for aggregate breadth. For one
instance and one budget the sequential engine remains the right tool. The fleet's value is
width per unit of wall clock — and, as the sweep below shows, how *cheap* that width is
depends entirely on instance size.


### Correctness gate

`verify` constructs replicas with a fixed seed and re-prices every constructed schedule with
the validated CPU engine, comparing to the deci-unit. **No fleet timing below is worth
reading until this passes.** The cell raises if the pricing diverges.


In [ ]:
if not globals().get('IG_FUTURE_READY', False):
    raise RuntimeError('Run the "Set up the environment" cell (section 1) first.')

import importlib.util

FLEET_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
_spec = importlib.util.spec_from_file_location('fleet_torch', ROOT / 'gpu' / 'fleet_torch.py')
fleet_torch = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(fleet_torch)

if FLEET_DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No CUDA GPU — the fleet runs on CPU with identical numbers, slowly.')

_t0 = time.perf_counter()
for _name in ('STC_NCOS_01', 'STC_NCOS_31'):
    _proc = subprocess.run(
        [sys.executable, str(ROOT / 'gpu' / 'fleet_torch.py'), 'verify',
         str(ROOT / 'masclib' / f'{_name}.csv'), '--device', FLEET_DEVICE],
        capture_output=True, text=True, check=False, timeout=300)
    _out = _proc.stdout.strip().splitlines()
    print(_out[-1] if _out else _proc.stderr.strip()[-300:])
    if _proc.returncode != 0 or 'MATCHES' not in _proc.stdout:
        raise AssertionError(f'Fleet pricing diverged from the validated engine on {_name}.')
TIMINGS['fleet · correctness gate'] = time.perf_counter() - _t0
print('Fleet pricing == engine pricing on this hardware.')


### How expensive is breadth?

The fleet's whole argument is that adding replicas is nearly free while the step is bound by
kernel-launch overhead rather than arithmetic. That is a measurable claim, so measure it:
hold the instance fixed, sweep R, and report the wall clock of one fleet iteration.

If the seconds per fleet-iteration stay flat as R grows, breadth is free and the ceiling is
launch overhead. When they start tracking R, the device is saturated and extra replicas cost
real time. Where that turning point sits depends on the instance, which is the honest answer
to "is the GPU worth it".


In [ ]:
if not globals().get('IG_FUTURE_READY', False):
    raise RuntimeError('Run the "Set up the environment" cell (section 1) first.')

SCALE_INSTANCES = ['STC_NCOS_15', 'STC_NCOS_31']     # n = 30, 75
SCALE_R = [64, 256, 1024] + ([4096] if FLEET_DEVICE == 'cuda' else [])
SCALE_ITERS = 5 if FLEET_DEVICE == 'cuda' else 2


def _sync():
    if FLEET_DEVICE == 'cuda':
        torch.cuda.synchronize()


_t0 = time.perf_counter()
_scale_rows, _scale_series = [], {}
for _name in SCALE_INSTANCES:
    _inst = Instance.parse(ROOT / 'masclib' / f'{_name}.csv')
    _scale_series[_name] = ([], [])
    _base = None
    for _R in SCALE_R:
        _fl = fleet_torch.Fleet(_inst, _R, d=2, dmax=0, device=FLEET_DEVICE, seed=1)
        _fl.construct(); _sync()
        _fl.iterate(1); _sync()                       # warm up kernels
        _t = time.perf_counter(); _fl.iterate(SCALE_ITERS); _sync()
        _per = (time.perf_counter() - _t) / SCALE_ITERS
        _base = _base or _per
        _scale_rows.append({
            'instance': _name, 'n': _inst.n, 'replicas (R)': _R,
            's / fleet-iteration': num(_per, 3),
            'cost vs R=64': f'{num(_per / _base, 2)}×',
            'replica-iterations/s': num(_R / _per, 0),
        })
        _scale_series[_name][0].append(_R)
        _scale_series[_name][1].append(_R / _per)
TIMINGS['fleet · scaling sweep'] = time.perf_counter() - _t0

display_table(_scale_rows, caption=f'Breadth sweep on {FLEET_DEVICE} — '
              'flat "cost vs R=64" means extra replicas are nearly free')

fig, axis = plt.subplots(figsize=(9.4, 4.0), constrained_layout=True)
for _i, (_name, (_xs, _ys)) in enumerate(_scale_series.items()):
    axis.plot(_xs, _ys, marker='o', color=[PALETTE['blue'], PALETTE['orange']][_i % 2], label=_name)
axis.set_xscale('log', base=2); axis.set_yscale('log')
axis.set_xlabel('replicas (R)'); axis.set_ylabel('replica-iterations / s')
axis.set_title('Throughput against fleet width')
axis.legend()
plt.show()


### Breadth versus depth, at one wall clock

Same instance, same total budget, spent two ways: **R replicas advancing together** against
**R sequential restarts** of the validated engine, each receiving `budget / R`. The fleet
prices exactly but searches differently (no swap phase, binomial removal), so the comparison
is about where the budget goes, not about pricing.


In [ ]:
if not globals().get('IG_FUTURE_READY', False):
    raise RuntimeError('Run the "Set up the environment" cell (section 1) first.')

COMPARE_INSTANCE = 'STC_NCOS_15'
COMPARE_SECONDS = 15
COMPARE_R = 512 if FLEET_DEVICE == 'cuda' else 32

_inst = Instance.parse(ROOT / 'masclib' / f'{COMPARE_INSTANCE}.csv')
_reference = float(benchmark[COMPARE_INSTANCE][1])
_seeds = random.Random(META_SEED).sample(range(2**32), COMPARE_R)

_fl = fleet_torch.Fleet(_inst, COMPARE_R, d=2, dmax=0, device=FLEET_DEVICE, seed=_seeds[0])
_fl.construct(); _sync()
_t0 = time.perf_counter()
while time.perf_counter() - _t0 < COMPARE_SECONDS:
    _fl.iterate(1)
_sync()
_fleet_elapsed = time.perf_counter() - _t0
_fleet_costs = sorted(int(c) / 10.0 for c in _fl.best_cost.cpu().tolist())

_budget = COMPARE_SECONDS / COMPARE_R
_t0 = time.perf_counter()
_seq_costs = sorted(solve(_inst, seconds=_budget, d=2, seed=int(s)).best_cost for s in _seeds)
_seq_elapsed = time.perf_counter() - _t0
TIMINGS['fleet · breadth vs depth'] = _fleet_elapsed + _seq_elapsed


def _spread(costs):
    q1, _, q3 = statistics.quantiles(costs, n=4)
    return {'best': num(min(costs), 2), 'median': num(statistics.median(costs), 2),
            'Q1–Q3': f'{num(q1, 2)}–{num(q3, 2)}',
            'reference hits': f'{sum(c <= _reference for c in costs)}/{len(costs)}',
            'distinct incumbents': len(set(costs))}


display_table([
    {'strategy': f'fleet · R={COMPARE_R} in lockstep', **_spread(_fleet_costs),
     'wall clock': f'{num(_fleet_elapsed, 1)} s',
     'throughput': f'{num(_fl.iterations * COMPARE_R / _fleet_elapsed, 0)} replica-iters/s'},
    {'strategy': f'{COMPARE_R} sequential restarts × {num(_budget, 3)} s', **_spread(_seq_costs),
     'wall clock': f'{num(_seq_elapsed, 1)} s', 'throughput': f'{COMPARE_R} restarts'},
], caption=f'{COMPARE_INSTANCE} · same {COMPARE_SECONDS}s wall clock · '
           f'recorded reference {num(_reference, 0)}')

fig, axis = plt.subplots(figsize=(9.4, 4.2), constrained_layout=True)
axis.hist(_seq_costs, bins=16, alpha=.65, color=PALETTE['muted'], label='sequential restarts')
axis.hist(_fleet_costs, bins=16, alpha=.65, color=PALETTE['blue'], label=f'fleet R={COMPARE_R}')
axis.axvline(_reference, color=PALETTE['red'], lw=1.4, ls='--', label='recorded reference')
axis.set_title('Incumbent distributions at equal wall clock')
axis.set_xlabel('final cost'); axis.set_ylabel('replicas / restarts'); axis.legend()
plt.show()


**Where the fleet earns its place.** Record hunting across thousands of simultaneous seeds;
parameter sweeps sharded across replicas; and scoring candidates for the factory in section 3,
where one candidate is exactly a handful of short independent runs. **Where it does not:**
CPU-only runtimes, deep search on a single instance, and large instances without the
iteration budget to use the width. The documented headroom is launch overhead —
`torch.compile`, CUDA graphs, and int32 cost accumulators (which fit for 49 of the 53
bundled instances; the four largest setup instances need the int64 fallback).


## 3. Direction B — LLM heuristic factory

In the lineage of FunSearch, EoH, ReEvo and the 2026 operator-ensemble work on flow shop, a
model proposes the Iterated Greedy **removal operator** as free-form code: which `d` jobs to
take out so the greedy repair rebuilds something cheaper. `factory/` scores each candidate on
a **training split** of instances and measures generalization on a **held-out test split**.
The baseline every candidate must beat is uniform random removal — a famously strong default
in the classic Iterated Greedy literature.

Two backends: an offline `local` mutation loop that needs no API key (it runs in CI, and it
is what the next cell executes), and an `llm` backend that calls an OpenAI-compatible
endpoint.

**The recorded null result.** A deterministic top-`d` removal — always taking the highest
scored jobs — measured **8.75%** mean gap against **4.15%** for random removal. Removing the
same jobs every iteration destroys the diversification the method depends on; the shipped
interface is therefore a standardized-softmax *sampler* that degenerates to exactly uniform
random when all scores are equal. Evolved operators then improved on the training split and
**lost to random on held-out seeds and instances**. That is the textbook failure mode of
LLM-driven heuristic design, reproduced here in miniature — and the bar any future campaign
has to clear.


In [ ]:
if not globals().get('IG_FUTURE_READY', False):
    raise RuntimeError('Run the "Set up the environment" cell (section 1) first.')

FACTORY_ARGS = ['--backend', 'local', '--gens', '2', '--pop', '4', '--seed', '7']
print('$ python factory/evolve.py ' + ' '.join(FACTORY_ARGS))

_t0 = time.perf_counter()
_proc = subprocess.run([sys.executable, str(ROOT / 'factory' / 'evolve.py'), *FACTORY_ARGS],
                       cwd=str(ROOT), capture_output=True, text=True, check=False, timeout=900)
TIMINGS['factory · offline campaign'] = time.perf_counter() - _t0
if _proc.returncode != 0:
    print(_proc.stderr[-2000:])
    raise AssertionError(f'factory/evolve.py exited with code {_proc.returncode}.')

for _ln in _proc.stdout.splitlines():
    if _ln.startswith(('baseline', 'gen ', 'heuristic:', 'train gap', 'test')) or 'winner' in _ln:
        print(_ln)

import re
_tr = re.search(r'train gap:\s*([\d.]+)%\s*\(random\s*([\d.]+)%\s*→\s*([+\-]?[\d.]+)pp\)', _proc.stdout)
_te = re.search(r'test\s+gap:\s*([\d.]+)%\s*\(random\s*([\d.]+)%\s*→\s*([+\-]?[\d.]+)pp\)', _proc.stdout)
if not (_tr and _te):
    raise AssertionError('Could not parse the train/test summary from the factory output.')
display_table([
    {'split': 'train (4 instances)', 'evolved operator': _tr.group(1) + '%',
     'random removal': _tr.group(2) + '%', 'Δ (pp)': _tr.group(3)},
    {'split': 'test (3 held-out)', 'evolved operator': _te.group(1) + '%',
     'random removal': _te.group(2) + '%', 'Δ (pp)': _te.group(3)},
], caption='Winner re-scored on fresh held-out seeds — a positive Δ would beat random removal')

print('\nAt this scale (a linear feature basis, four training instances) the evolved operator '
      'is not expected to win. What the run demonstrates is the loop: multi-seed scoring, an '
      'honest re-evaluation on fresh seeds, and a null result that reproduces.')


### Where a real model plugs in

The `llm` backend posts to any OpenAI-compatible endpoint and compiles the returned operator
in a restricted namespace, smoke-testing it before it can score:

```bash
cp .env.example .env      # OPENAI_API_KEY, OPENAI_BASE_URL, OPENAI_MODEL
set -a; . ./.env; set +a
python factory/evolve.py --backend llm --gens 6 --pop 8
```

What the literature says a serious campaign needs, and this one deliberately does not have:
free-form programs rather than a linear basis, a *set* of complementary operators rather than
a single champion, splits separated by scale **and** distribution, at least five seeds per
candidate during evolution with fresh seeds at the final evaluation, and a success threshold
registered against random removal *before* the campaign starts. The evaluator's wall clock —
not the tokens — is the binding constraint, which is why the fast engines in this repository
exist at all.


## 4. Reproducibility manifest

Everything this run used: repository origin, versions, the device the fleet landed on, the
wall clock of each part, and a UTC stamp. Recomputed on every execution.


In [ ]:
if not globals().get('IG_FUTURE_READY', False):
    raise RuntimeError('Run the "Set up the environment" cell (section 1) first.')

import datetime
_commit = 'unknown'
if (ROOT / '.git').exists():
    _commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT,
                             capture_output=True, text=True, check=False).stdout.strip() or 'unknown'
print(json.dumps({
    'repository': str(ROOT),
    'commit': _commit,
    'environment': 'Google Colab' if IN_COLAB else 'local',
    'python': platform.python_version(),
    'torch': torch.__version__,
    'fleet device': FLEET_DEVICE,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'seconds': {k: round(v, 1) for k, v in TIMINGS.items()},
    'total seconds': round(sum(TIMINGS.values()), 1),
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}, indent=2))
